## Problem 0: GitHub Issues

Progress is tracked in this repo's [Issues](https://github.com/qahramantaha/computational-theory/issues): one issue per problem, and issues are closed when each problem is done.

## Problem 1: Representing SHA-256 Data

### 32-bit words
A word is 32 bits (section 2.1) and when you add words you do it mod 2³² (section 3.2). Python ints can be any size, so I use a normal int and mask it with `0xFFFFFFFF` to keep it at 32 bits. I put this in `add32` since I'll need it again later. The last few prints check my setup against the examples in section 3.1.

In [7]:
MASK32 = 0xFFFFFFFF   # all 32 bits on, biggest number a word can be

def add32(x, y):
    # adding two words, keeps result at 32 bits (section 3.2)
    return (x + y) & MASK32

w = 0xFFFFFFFF
print(w + 1)                          # python just keeps going past 32 bits
print(add32(w, 1))                    # with the mask it wraps back to 0
print(add32(w, 1) == (w + 1) % 2**32) # masking is the same as mod 2^32

# checking against the examples in section 3.1
print(hex(int("10100001000000111111111000100011", 2)))  # should be a103fe23
print(f"{291:08x}")                   # should be 00000123
print(f"{0x6a09e667:032b}")           # a word as 32 bits

4294967296
0
True
0xa103fe23
00000123
01101010000010011110011001100111


### Sequences of 32-bit words
I use a list of ints. It keeps the order and I can index it like `W[t]`, same as the standard does.

In [8]:
# SHA-256 initial hash value H(0), section 5.3.3
H0 = [0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
      0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19]

print(len(H0))
print(hex(H0[0]))

8
0x6a09e667


### Input messages
Messages are stored as `bytes`. Text gets turned into bytes with `.encode("utf-8")`, and each byte is a number from 0 to 255. The standard's own example uses "abc" (section 5.1.1), so I use the same one to check.

In [9]:
msg = "abc".encode("utf-8")
print(msg)
print(list(msg))                       # each byte as a number
print([f"{b:08b}" for b in msg])       # should match section 5.1.1: 01100001 01100010 01100011
print(len(msg) * 8)                    # length in bits, should be 24

b'abc'
[97, 98, 99]
['01100001', '01100010', '01100011']
24


### 512-bit message blocks
A block is 512 bits, which is 64 bytes, and gets split into 16 words (sections 3.1 and 5.2.1). I just use 64 made-up bytes here to show the split. Real blocks come from padding, which gets done later.

In [10]:
block = bytes(range(64))    # 64 made-up bytes (0 to 63) = 512 bits

words = [int.from_bytes(block[i:i+4], "big") for i in range(0, 64, 4)]

print(len(block) * 8)       # should be 512
print(len(words))           # should be 16
print([hex(x) for x in words[:3]])  # first 3 words

512
16
['0x10203', '0x4050607', '0x8090a0b']


### Final 256-bit hash value
The hash is 8 words stuck together (section 6.2), so 8 × 32 = 256 bits. I keep it as a list of 8 ints while working, then turn each word into 4 bytes and join them. I'm using H(0) here just as an example of 8 words. The `hashlib` line is only there to show what a real SHA-256 output looks like.

In [11]:
import hashlib

# using H(0) just as an example of 8 words
hash_bytes = b"".join(x.to_bytes(4, byteorder="big") for x in H0)

print(len(hash_bytes) * 8)     # should be 256
print(hash_bytes.hex())        # shows the 32 bytes as hex text
print(len(hash_bytes.hex()))   # 64 hex characters

# real SHA-256 of "abc", same size and format
real = hashlib.sha256(b"abc").digest() #raw 32 bytes
print(len(real) * 8)           # also 256
print(real.hex())              # Shows the real hash

256
6a09e667bb67ae853c6ef372a54ff53a510e527f9b05688c1f83d9ab5be0cd19
64
256
ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


### Big-endian integers
SHA-256 is big-endian (section 3.1), means the most important byte comes first. Section 3.1 uses 291 as an example, which should be the word `00000123`. Python lets us pick the byte order, so I always pass `"big"`. Using `"little"` flips the bytes and gives a different number. This matters because the message length added in padding is stored big-endian too (section 5.1.1).

In [12]:
# 291 from section 3.1
print((291).to_bytes(4, "big").hex())       # big-endian hex representation 
print((291).to_bytes(4, "little").hex())    # little-endian hex representation

b = bytes.fromhex("00000123")              # turn hex back into bytes
print(int.from_bytes(b, "big"))            # back to 291
print(int.from_bytes(b, "little"))         # wrong: 587268096

00000123
23010000
291
587268096
